# reasoning.py 真实场景测试

针对 `app/agent/reasoning.py`（route_intent 节点的"大脑"）的**真实链路**测试：
**真实数据库会话 + 真实 LLM 路由器（react 模式）**，不打桩、不 Fake。

所有 case 均模拟 HR Agent 现有业务场景（与前端 `HRAgent.vue` 对话框、
`/agent/graph/chat` 端点进入 `route_intent_node` 的输入一致）。

| 章节 | 场景 | 是否调 LLM | 是否写库 |
|---|---|---|---|
| §1 | 规则快通道：JD 生成/编辑、评分标准编辑、删除、简历筛选、试卷、面试、邮件 | 否 | 否（只读） |
| §2 | confirmed_requirements 确认回传直通（JD / send_email skill） | 否 | 否 |
| §3 | 真实会话记忆构建（库里已有的 JD 生成对话） | 否 | 否（只读） |
| §4 | 观察型 ReAct 路由：闲聊 / 资源盘点 / 结合记忆回顾 / 模糊招聘需求 | **是** | 否（只读） |
| §5 | AGENT_ROUTER_MODE=rule 与 react 对比 | 否 | 否 |
| §6 | 删除候选追问（真实 JD 列表构造候选清单 + 指代/名称继承 + 防劫持回归） | 否 | 否（只读） |
| §7 | search_resources 只读观察工具（路由器看到的世界） | 否 | 否（只读） |
| §8 | route_intent_node 整节点执行（记忆+决策+dest_node 分发） | 是 | 否（只读） |

> ⚠️ **成本提示**：§4/§4.1/§8 共约 7 次真实路由 LLM 调用（每次约 5–15s，消耗模型配额）。
> 整个 Notebook **不写数据库、不删除任何资源**——删除场景只测"路由决策"，不执行删除。
> 运行前提：`backend/.env` 已配置 `LLM_API_KEY`、`DATABASE_URL` 可连通。


## §0 环境准备：真实配置 / 真实 DB / 真实用户

In [1]:
import os, sys, time, json
from pathlib import Path

# 定位 backend 根目录（notebook 在 backend/examples/ 下）
BACKEND = Path.cwd()
if BACKEND.name == "examples":
    BACKEND = BACKEND.parent
assert (BACKEND / "app" / "agent" / "reasoning.py").exists(), "请从 backend/ 或 backend/examples/ 启动"
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))

from app.core.config import settings
settings.DEBUG = False  # 关闭 SQL echo（必须在 import database 之前）

from app.agent import reasoning
from app.agent.llm import chat_llm_available

print("AGENT_ROUTER_MODE      =", settings.AGENT_ROUTER_MODE, "->", reasoning.router_mode())
print("路由模型               =", settings.AGENT_ROUTER_MODEL or settings.LLM_MODEL)
print("路由超时/递归上限      =", settings.AGENT_ROUTER_TIMEOUT_SECONDS, "s /", settings.AGENT_ROUTER_RECURSION_LIMIT)
print("LLM 可用               =", chat_llm_available())
assert chat_llm_available(), "未配置 LLM_API_KEY：§4/§8 的真实 ReAct 路由将无法演示"
assert reasoning.router_mode() == "react", "本 Notebook 演示真实 react 路由，请保持 AGENT_ROUTER_MODE=react（默认）"

AGENT_ROUTER_MODE      = react -> react
路由模型               = qwen3.8-flash
路由超时/递归上限      = 20.0 s / 8
LLM 可用               = True


In [2]:
from sqlalchemy import select, func
from app.core.database import AsyncSessionLocal
from app.models.conversation import Conversation, Message
from app.models.user import User
from app.services.agent_service import AgentService

db = AsyncSessionLocal()  # 真实 AsyncSession（全程只读）

# 选库里消息最多的真实用户（当前环境为 testuser，有真实 JD 生成对话）
row = (await db.execute(
    select(Conversation.user_id, func.count(Message.id).label("n"))
    .join(Message, Message.conversation_id == Conversation.id)
    .group_by(Conversation.user_id).order_by(func.count(Message.id).desc()).limit(1)
)).first()
assert row, "库里没有任何对话消息：请先在 HR Agent 前端真实聊几轮再运行"
USER_ID = row[0]
user = (await db.execute(select(User).where(User.id == USER_ID))).scalars().first()
print("真实用户:", user.username, USER_ID, "| 消息数:", row[1])

service = AgentService(db)          # 真实服务：真实 tool_registry / skill_dispatcher / intent_service
service._graph_artifacts = []       # 图内累积产物（route_intent_node  normally 由 state 注入）
print("tool_registry:", sorted(service.tool_registry.keys()))
print("skill bundles:", {k: v.metadata.confirmation_action for k, v in service.skill_dispatcher.bundles.items()})

真实用户: testuser fb04aeeb-e2e5-48cc-9308-b083a4049c0b | 消息数: 18
tool_registry: ['criteria_edit', 'email_notification', 'exam_generate', 'interview_plan', 'jd', 'jd_edit', 'resource_delete', 'resume_screening']
skill bundles: {'email_notification': 'send_email'}


## §1 规则快通道（零 LLM）

`resolve_intent_and_plan(user_id=None)` 强制走本地规则——这正是生产上
"明确命中 HR 关键词时不花 LLM 成本"的快通道。每条 case 对应一个真实业务入口：

判定优先级：**删除 > 评分标准编辑 > JD 编辑 > fast 关键词 > 附件上下文 > 追问循环 > 兜底正则**。

In [3]:
RESUME_ATTS = service._normalize_attachments([
    {"name": "张三_后端开发.pdf", "size": 204800, "content_type": "application/pdf"},
    {"name": "李四_Java工程师.docx", "size": 102400, "content_type": "application/vnd.openxmlformats-officedocument.wordprocessingml.document"},
])
EXAM_ATTS = service._normalize_attachments([
    {"name": "Java基础培训手册.pdf", "size": 512000, "content_type": "application/pdf"},
])

# (场景, 用户消息, 附件, 预期 intent, 预期 source)
RULE_CASES = [
    ("JD 生成·完整需求",   "帮我生成一个Java开发工程师的JD，base杭州，3-5年经验，本科，薪资20-35K", [], "jd", "rule_intent"),
    ("JD 生成·口语化",     "我们要招一个前端开发，帮我写一份职位描述", [], "jd", "rule_intent"),
    ("JD 编辑·带指代",     "把上次那个JD的薪资改成25-40K，再加上远程办公福利", [], "jd_edit", "jd_edit_rule"),
    ("评分标准编辑",       "修改一下简历评分标准，把学历权重降低，项目经验加分提高", [], "criteria_edit", "criteria_edit_rule"),
    ("资源删除",           "删除刚才生成的产品经理JD", [], "resource_delete", "delete_rule"),
    ("简历筛选·附件上下文", "帮我筛选一下这几份简历，给候选人评分", RESUME_ATTS, "resume_screening", "rule_intent"),
    ("试卷生成·附件上下文", "根据这份培训手册出一套笔试题，30道选择题", EXAM_ATTS, "exam_generate", "rule_intent"),
    ("面试方案",           "给通过筛选的候选人安排面试方案", [], "interview_plan", "rule_intent"),
    ("面试邀约邮件(skill)", "给候选人发一封面试邀约邮件", [], "email_notification", "rule_intent"),
    ("筛选循环追问",       "把评分大于60的候选人推进到面试环节", [], "interview_plan", "rule_intent"),
    ("面试→试卷追问",      "就按当前面试方案出一套试卷", [], "exam_generate", "rule_intent"),
]

print(f"{'场景':<16} {'intent':<18} {'source':<20} 结果")
print("-" * 78)
for label, msg, atts, want_intent, want_source in RULE_CASES:
    intent, plan = await reasoning.resolve_intent_and_plan(
        service, msg, atts, memory_context="", confirmed_requirements=None, user_id=None,
    )
    ok = intent == want_intent and plan.get("source") == want_source
    print(f"{label:<16} {intent:<18} {plan.get('source'):<20} {'✅' if ok else '❌ 预期 ' + want_intent + '/' + want_source}")
    assert ok, (label, intent, plan)

场景               intent             source               结果
------------------------------------------------------------------------------
JD 生成·完整需求       jd                 rule_intent          ✅
JD 生成·口语化        jd                 rule_intent          ✅
JD 编辑·带指代        jd_edit            jd_edit_rule         ✅
评分标准编辑           criteria_edit      criteria_edit_rule   ✅
资源删除             resource_delete    delete_rule          ✅
简历筛选·附件上下文       resume_screening   rule_intent          ✅
试卷生成·附件上下文       exam_generate      rule_intent          ✅
面试方案             interview_plan     rule_intent          ✅
面试邀约邮件(skill)    email_notification rule_intent          ✅
筛选循环追问           interview_plan     rule_intent          ✅
面试→试卷追问          exam_generate      rule_intent          ✅


In [4]:
# 规则未命中的消息：build_rule_agent_plan 返回 None → 生产上交给 ReAct（§4 演示）
misses = ["你好，你能帮我做些什么？", "帮我看看我最近都生成过哪些东西", "我想招个人"]
for msg in misses:
    plan = reasoning.build_rule_agent_plan(service, msg, [], memory_context="", artifacts=[])
    print(f"{msg!r:36} -> rule plan = {plan}")
    assert plan is None

'你好，你能帮我做些什么？'                       -> rule plan = None
'帮我看看我最近都生成过哪些东西'                    -> rule plan = None
'我想招个人'                              -> rule plan = None


## §2 confirmed_requirements 确认回传直通

用户在 JD 确认弹窗点"确认"后，前端带 `confirmed_requirements` 再次请求
（`/agent/graph/chat`）。此时**完全跳过意图决策**：
- 无 action → `jd`（source=`confirmed_requirements`）
- `action=send_email` 命中 skill → `email_notification`（source=`confirmed_skill_action`）

In [5]:
# 真实 JD 确认回传（字段来自 jd_prepare 弹窗）
intent, plan = await reasoning.resolve_intent_and_plan(
    service, "已确认招聘信息，请生成 JD。", [], "",
    confirmed_requirements={"job_title": "Java开发工程师", "location": "杭州", "salary": "20-35K",
                            "experience": "3-5年", "education": "本科"},
    user_id=USER_ID,
)
print("JD 确认:", intent, "|", plan["source"], "|", plan["reason"])
assert intent == "jd" and plan["source"] == "confirmed_requirements"

# 真实邮件 skill 确认回传（skills/hr-agent-email/skill.json: confirmation_action=send_email）
intent, plan = await reasoning.resolve_intent_and_plan(
    service, "", [], "",
    confirmed_requirements={"action": "send_email", "to": "zhangsan@example.com", "subject": "面试邀约"},
    user_id=USER_ID,
)
print("Skill 确认:", intent, "|", plan["source"], "|", plan["reason"])
assert intent == "email_notification" and plan["source"] == "confirmed_skill_action"

JD 确认: jd | confirmed_requirements | 用户已确认 JD 生成信息，继续执行 JD 工具链。
Skill 确认: email_notification | confirmed_skill_action | 用户已确认 hr-agent-email 所需信息，继续执行 skill。


## §3 真实会话记忆构建

`build_conversation_memory` 从库里读最近消息（当前环境里是真实的
"AI RAG 开发 JD 生成"对话）。观察：角色标签、当前消息去重、条数/长度截断。

In [6]:
# 取该用户消息最多的真实会话
conv_row = (await db.execute(
    select(Conversation.id, Conversation.title, func.count(Message.id).label("n"))
    .join(Message, Message.conversation_id == Conversation.id)
    .where(Conversation.user_id == USER_ID)
    .group_by(Conversation.id).order_by(func.count(Message.id).desc()).limit(1)
)).first()
CONV_ID = str(conv_row[0])
print("真实会话:", CONV_ID, "|", conv_row[1], "| 消息数:", conv_row[2])

memory = await reasoning.build_conversation_memory(service, CONV_ID, USER_ID, "帮我回顾一下刚才我们聊的内容")
print(f"\n--- memory_context ({len(memory)} chars) ---")
print(memory)

真实会话: d5605703-807c-4393-b810-76422a8ec18e | 给我生成一个 AI RAG 开发的 jd,base 南京，薪 | 消息数: 6

--- memory_context (1088 chars) ---
用户: 给我生成一个 AI RAG 开发的 jd,base 南京，薪资15k
助手: 我先补齐 JD 生成的必要信息，再生成会更稳。请在弹窗里确认或补充这些字段。
用户: 给我生成一个 AI RAG 开发的 jd,base 南京，薪资15k
助手: 我先补齐 JD 生成的必要信息，再生成会更稳。请在弹窗里确认或补充这些字段。
用户: 已确认招聘信息，请生成 JD。
助手: ## 岗位 JD # AI RAG 开发 ## 岗位信息 - **岗位名称：** AI RAG 开发 - **所属部门：** IT部 - **工作地点：** 南京 - **薪资范围：** 15k - **经验要求：** 3年以上 - **学历要求：** 本科 ## 岗位职责 1. 负责基于大模型的 RAG（检索增强生成）系统的设计、开发与优化，提升问答与内容生成的准确性和可靠性； 2. 负责 RAG 链路中各环节的工程实现，包括文档解析、文本切分、向量化、检索召回、重排序及生成效果调优； 3. 负责向量数据库的选型、部署、索引设计与性能优化，保障检索服务的高效稳定运行； 4. 负责大模型接口的集成与调用优化，包括 Prompt 设计、上下文管理及生成效果评估； 5. 参与 RAG 相关系统的架构设计、技术方案评审与代码评审，保障系统质量与可维护性； 6. 跟踪大模型与 RAG 领域前沿技术，推动新技术在业务场景中的落地应用。 ## 任职要求 1. 本科及以上学历，计算机、软件工程、人工智能等相关专业优先； 2. 3年以上开发经验，具备大模型应用或 RAG 系统相关开发经验； 3. 熟悉大模型（LLM）原理及应用开发，具备 Prompt 工程与生成效果调优经验； 4. 熟悉 RAG 技术架构，掌握文档解析、文本切分、向量化、检索召回与重排序等关键环节； 5. 熟悉主流向量数据库（如 Milvus、Faiss、Weaviate、Qdrant 等）的使用与性能优化； 6. 具备扎实的编程能力，熟悉 Python 等开发语言，有良好的工程实现与代码规范意识； 7. 

In [7]:
# 细节验证：
# 1) 当前消息去重：把会话里第一条用户消息当作"当前消息"，它不应出现在记忆里
first_user_msg = (await db.execute(
    select(Message.content).where(Message.conversation_id == conv_row[0], Message.role == "user")
    .order_by(Message.created_at).limit(1)
)).scalars().first()
mem_dedup = await reasoning.build_conversation_memory(service, CONV_ID, USER_ID, first_user_msg)
print("当前消息去重生效:", first_user_msg[:30] not in mem_dedup or mem_dedup.count(first_user_msg[:30]) < memory.count(first_user_msg[:30]))

# 2) limit 截断：只取最近 2 条
mem_short = await reasoning.build_conversation_memory(service, CONV_ID, USER_ID, "x", limit=2)
print("limit=2 行数:", len(mem_short.splitlines()), "(默认12:", len(memory.splitlines()), ")")

# 3) 非 UUID / 不存在会话 → 静默返回空串（不阻断路由）
print("非UUID:", repr(await reasoning.build_conversation_memory(service, "sandbox-thread", USER_ID, "x")))
print("他人会话:", repr(await reasoning.build_conversation_memory(service, CONV_ID, __import__("uuid").uuid4(), "x")))

当前消息去重生效: True
limit=2 行数: 5 (默认12: 6 )
非UUID: ''
他人会话: ''


## §4 观察型 ReAct 路由（真实 LLM）

规则未命中的消息进入 `plan_agent_action` → `routing_agent.run_router_agent`：
`create_agent` + 只读工具 `search_resources` + `ToolStrategy(RouterDecision)` 结构化输出。
模型只能在 **chat / use_tool / ask_user** 中选择，intent 必须在 tool_registry 白名单内。

每次调用约 5–15s。观察 `source=react_router`、thought、observation、confidence。

In [8]:
async def react_case(label, msg, memory_context="", conversation_id=None, attachments=None):
    t0 = time.perf_counter()
    intent, plan = await reasoning.resolve_intent_and_plan(
        service, msg, attachments or [], memory_context=memory_context,
        confirmed_requirements=None, user_id=USER_ID, conversation_id=conversation_id,
        artifacts=service._graph_artifacts,
    )
    dt = time.perf_counter() - t0
    print(f"\n{'='*70}\n【{label}】{msg!r}  ({dt:.1f}s)")
    print(f"  intent={intent}  action={plan.get('action')}  mode={plan.get('mode')}  "
          f"confidence={plan.get('confidence')}  source={plan.get('source')}")
    print(f"  thought: {plan.get('reason')}")
    if plan.get("observation"):
        print(f"  observation: {str(plan.get('observation'))[:200]}")
    if plan.get("reply"):
        print(f"  reply: {str(plan.get('reply'))[:200]}...")
    return intent, plan

# A. 闲聊/能力询问 → 预期 chat + general
intent_a, plan_a = await react_case("A 闲聊", "你好，你能帮我做些什么？")

# B. 资源盘点（指代"最近生成的东西"）→ 预期模型调用 search_resources 观察后 chat
intent_b, plan_b = await react_case("B 资源盘点", "帮我看看我最近都生成过哪些东西")

# C. 结合真实会话记忆回顾 → 预期 chat，reply 提到记忆里的真实岗位（AI RAG / Java）
intent_c, plan_c = await react_case("C 记忆回顾", "帮我回顾一下刚才我们聊的内容",
                                    memory_context=memory, conversation_id=CONV_ID)

# D. 模糊招聘需求（无关键词命中）→ 观察模型如何决策（ask_user 追问 / jd 进工具链）
intent_d, plan_d = await react_case("D 模糊需求", "我想招个人")


【A 闲聊】'你好，你能帮我做些什么？'  (7.1s)
  intent=general  action=chat  mode=chat  confidence=None  source=react_router
  thought: 用户只是打招呼并询问能力范围，无具体 HR 任务，直接对话回复。
  observation: ReAct 路由已查询工具观察现状。
  reply: 你好！我是你的 HR 智能助手，可以帮你完成招聘流程中的这些工作：

1. **生成岗位 JD**：告诉我岗位名称、地点、薪资、经验、学历要求，我来输出 JD，并同步生成配套的简历评分标准。
2. **修改 JD / 评分标准**：对最近生成的 JD 或评分标准做调整，比如补充职责、改薪资区间、调整权重。
3. **简历筛选评分**：上传 PDF/DOC/DOCX 简历，选择对应 JD，我会批量打...


ReAct 路由未产出 structured_response（第 1/2 次）


ReAct 路由失败，回退本地规则: 



【B 资源盘点】'帮我看看我最近都生成过哪些东西'  (31.2s)
  intent=general  action=chat  mode=chat  confidence=None  source=fallback
  thought: ReAct 路由不可用后，使用本地关键词和附件规则选择下一步。
  observation: 未匹配到需要调用的工具。



【C 记忆回顾】'帮我回顾一下刚才我们聊的内容'  (8.9s)
  intent=general  action=chat  mode=chat  confidence=0.96  source=react_router
  thought: 用户只是要求回顾对话内容，属于总结性普通对话，无需调用业务工具。
  observation: 历史对话：用户要求生成 AI RAG 开发 JD（base 南京、薪资 15k），确认招聘信息后已生成完整 JD 及配套的简历初筛评分标准。
  reply: 好的，帮你梳理一下我们刚才的进展：

## 一、你的需求
生成一份 **AI RAG 开发** 岗位的招聘 JD，关键条件：
- 工作地点：**南京**
- 薪资：**15k**

## 二、已完成的内容

**1. 岗位 JD（已生成）**
- 岗位名称：AI RAG 开发
- 所属部门：IT部
- 地点/薪资/经验/学历：南京 / 15k / 3年以上 / 本科
- 岗位职责 6 条：RAG ...



【D 模糊需求】'我想招个人'  (6.4s)
  intent=jd  action=ask_user  mode=tool  confidence=0.72  source=react_router
  thought: 用户想启动招聘，最直接对应生成岗位 JD，但岗位名称、地点、薪资、经验、学历等前置信息均未提供，需先补齐。
  observation: 当前线程无已生成产物（JD/评分标准/简历/面试方案/试卷均为 0），附件为空，无法定位已有岗位。
  reply: 好的，先帮你把这个岗位定义出来。请补充几项关键信息：岗位名称、工作地点、薪资范围、经验要求、学历要求（有招聘经理或团队规模也可以一并说明）。信息齐了我再生成 JD，并同步产出配套的简历评分标准。...


In [9]:
# ReAct 决策合法性校验（reasoning.plan_agent_action 的白名单保险）
allowed = set(reasoning._allowed_intents(service))
for label, plan in [("A", plan_a), ("B", plan_b), ("C", plan_c), ("D", plan_d)]:
    assert plan["intent"] in allowed, (label, plan["intent"])
    assert plan["action"] in {"chat", "use_tool", "ask_user"}
    assert plan["source"] in {"react_router", "fallback"}  # LLM 偶发失败时静默回退规则，不硬失败
    print(label, "✅", plan["intent"], plan["action"], plan["source"])

A ✅ general chat react_router
B ✅ general chat fallback
C ✅ general chat react_router
D ✅ jd ask_user react_router


### §4.1 回退复盘：为什么 B/C 可能走 fallback？

真实运行中常见两种回退（都是 reasoning 多层兜底设计在起作用，**不是 bug**）：

1. **`Recursion limit of 8 reached`**：模型对"盘点所有资源"会连续调用
   `search_resources`（jd→resume→interview→exam…），每次工具调用消耗 2 步递归，
   默认 `AGENT_ROUTER_RECURSION_LIMIT=8` 不够用 → 异常被 `plan_agent_action`
   捕获 → 回退本地规则。
2. **`未产出 structured_response`**：模型偶发没有按 `ToolStrategy(RouterDecision)`
   输出结构化决策，`run_router_agent` 重试 `AGENT_ROUTER_MAX_ATTEMPTS=2` 次后放弃 → 回退。

下面把递归上限临时调到 16 重试 B/C，观察成功的 ReAct 观察链路（配置在调用时读取，
`routing_agent.py:153/156`，改完即生效，finally 恢复）。

In [10]:
ORIGINAL_LIMIT = settings.AGENT_ROUTER_RECURSION_LIMIT
try:
    settings.AGENT_ROUTER_RECURSION_LIMIT = 16
    intent_b2, plan_b2 = await react_case("B 重试·递归上限16", "帮我看看我最近都生成过哪些东西")
    intent_c2, plan_c2 = await react_case("C 重试·递归上限16", "帮我回顾一下刚才我们聊的内容",
                                          memory_context=memory, conversation_id=CONV_ID)
finally:
    settings.AGENT_ROUTER_RECURSION_LIMIT = ORIGINAL_LIMIT
print("\n已恢复递归上限:", settings.AGENT_ROUTER_RECURSION_LIMIT)
print("提示：若生产上资源盘点类请求频繁回退，可在 .env 调高 AGENT_ROUTER_RECURSION_LIMIT")

ReAct 路由未产出 structured_response（第 1/2 次）



【B 重试·递归上限16】'帮我看看我最近都生成过哪些东西'  (35.8s)
  intent=general  action=chat  mode=chat  confidence=None  source=react_router
  thought: 用户只想查看历史生成记录，已通过只读查询汇总，无需业务工具。
  observation: 已查到 1 个 JD、1 套评分标准、1 条简历评分记录；面试方案与试卷为空。
  reply: 帮你翻了一下最近的生成记录，目前一共有这些：

**岗位 JD**
- AI RAG 开发 JD

**简历评分标准**
- AI RAG 开发评分标准（随 JD 自动生成）

**简历评分记录**
- 1 条候选人评分记录（名称未识别，可能简历文件名缺失）

**面试方案**
- 暂无

**考试试卷**
- 暂无

需要的话我可以：把那条简历评分记录的候选人详情调出来、基于它生成面试方案，或者...



【C 重试·递归上限16】'帮我回顾一下刚才我们聊的内容'  (8.9s)
  intent=general  action=chat  mode=chat  confidence=0.95  source=react_router
  thought: 用户只是要回顾对话内容，属于普通对话总结，不需要执行业务工具。
  observation: ReAct 路由已查询工具观察现状。
  reply: 好的，帮你回顾一下我们刚才的对话：

## 1. 你的需求
你提出：生成一个 **AI RAG 开发** 岗位的 JD，**Base 南京**，**薪资 15k**。

## 2. 我的处理
我先提示补齐 JD 生成的必要字段（岗位名称、地点、薪资、经验、学历），你在弹窗中确认后，我完成了生成。

## 3. 已产出的内容

**岗位 JD 关键信息**
- 岗位名称：AI RAG 开发
- 所属...

已恢复递归上限: 8
提示：若生产上资源盘点类请求频繁回退，可在 .env 调高 AGENT_ROUTER_RECURSION_LIMIT


## §5 路由模式开关对比：rule vs react

同一条模糊消息，`AGENT_ROUTER_MODE=rule` 时直接走本地规则兜底
（source=`fallback`，零 LLM 成本）——评测/离线调试推荐模式。

In [11]:
ORIGINAL_MODE = settings.AGENT_ROUTER_MODE
try:
    settings.AGENT_ROUTER_MODE = "rule"
    intent, plan = await reasoning.resolve_intent_and_plan(
        service, "你好，你能帮我做些什么？", [], "", confirmed_requirements=None, user_id=USER_ID,
    )
    print("rule 模式:", intent, "|", plan["source"], "|", plan["reason"])
    assert plan["source"] == "fallback"
finally:
    settings.AGENT_ROUTER_MODE = ORIGINAL_MODE
print("已恢复:", reasoning.router_mode())

rule 模式: general | fallback | ReAct 路由不可用后，使用本地关键词和附件规则选择下一步。
已恢复: react


## §6 删除候选追问（真实 JD 数据）

真实场景：用户说"删除Java的JD" → 删除链查到 **2 个同名 JD** → 列出候选清单
（`delete_resource_candidates` artifact）→ 用户回"就删最新的那个吧"或直接回名称。

这里用**真实 `_list_deletable_resources` + `_match_delete_resources`** 构造与
`nodes/resources.py` 完全一致的候选 artifact，只测 reasoning 的继承规则，**不执行删除**。

In [12]:
from app.schemas.agent import AgentArtifact

real_jds = await service._list_deletable_resources("jd", USER_ID)
print("真实 JD 资源:", [(j["name"], j["id"][:8]) for j in real_jds])
matches = service._match_delete_resources(real_jds, "Java", latest=False)
print("关键词 'Java' 匹配:", len(matches), "个")

if len(matches) > 1:
    # 与 nodes/resources.py 多候选分支完全一致的 artifact 构造
    candidates_artifact = AgentArtifact(
        type="delete_resource_candidates", title="待确认删除目标",
        content={"resource_type": "jd", "keyword": "Java", "candidates": matches[:8]},
    ).model_dump()
    service._graph_artifacts = [candidates_artifact]

    for msg in ["就删最新的那个吧", matches[0]["name"], "删除第一个"]:
        inherited = service._inherit_delete_type_from_candidates(service._graph_artifacts, msg)
        intent, plan = await reasoning.resolve_intent_and_plan(
            service, msg, [], "", confirmed_requirements=None, user_id=None,
            artifacts=service._graph_artifacts,
        )
        print(f"\n追问 {msg!r}: inherit={inherited} -> intent={intent} source={plan.get('source')}")

    # 防劫持回归：候选清单存在时，明确的新建意图不能被抢进删除流程
    intent, plan = await reasoning.resolve_intent_and_plan(
        service, f"帮我生成一个{matches[0]['name']}的JD", [], "", confirmed_requirements=None,
        user_id=None, artifacts=service._graph_artifacts,
    )
    print(f"\n防劫持回归: intent={intent} source={plan.get('source')}")
    assert intent == "jd" and plan["source"] == "rule_intent"
else:
    print("⚠️ 当前用户同名 JD 不足 2 个，跳过候选追问演示（可先在前端真实生成两个同名 JD）")

真实 JD 资源: [('Java开发工程师', 'ee746633'), ('Java开发工程师', '5b7e7daa')]
关键词 'Java' 匹配: 2 个

追问 '就删最新的那个吧': inherit=('jd', False) -> intent=resource_delete source=delete_followup_rule

追问 'Java开发工程师': inherit=('jd', True) -> intent=resource_delete source=delete_followup_rule

追问 '删除第一个': inherit=('jd', False) -> intent=resource_delete source=delete_followup_rule

防劫持回归: intent=jd source=rule_intent


## §7 search_resources：路由器看到的世界

ReAct 路由唯一的观察工具（只读）。直接调用它，看模型决策时拿到的真实资源快照
（≤5 条、id 前 8 位、名称 ≤80 字符、JSON ≤1200 字符）。

In [13]:
from langchain_core.runnables import RunnableConfig
from app.agent.tools.router_tools import search_resources, ROUTER_TOOLS

print("路由器工具清单:", [t.name for t in ROUTER_TOOLS], "（只有只读观察，无写工具）")
cfg = RunnableConfig(configurable={"agent_service": service, "user_id": USER_ID, "conversation_id": CONV_ID})
for rt in ["jd", "resume", "interview", "exam"]:
    obs = await search_resources.ainvoke({"resource_type": rt, "keyword": ""}, config=cfg)
    print(f"\n[{rt}] {obs}")

路由器工具清单: ['search_resources'] （只有只读观察，无写工具）



[jd] {"resource_type": "jd", "keyword": "", "items": [{"id": "7185f06a", "name": "AI RAG 开发 JD", "updated_at": ""}]}



[resume] {"resource_type": "resume", "keyword": "", "items": [{"id": "04fefe0d", "name": "未知", "updated_at": ""}]}



[interview] {"resource_type": "interview", "keyword": "", "items": []}

[exam] {"resource_type": "exam", "keyword": "", "items": []}


## §8 route_intent_node 整节点执行

reasoning 的唯一生产调用方。用真实 state + config 直接执行节点，
观察完整输出：`intent / agent_plan / route / memory_context / dest_node`。
`dest_node` 就是主图条件边 `_route_destination` 的分发目标。

In [14]:
from app.agent.nodes.routing import route_intent_node

async def run_node(msg, conversation_id=None):
    state = {
        "user_message": msg, "conversation_id": conversation_id,
        "attachments": [], "artifacts": service._graph_artifacts,
        "confirmed_requirements": None, "agent_plan": None, "preset_intent": None,
    }
    config = RunnableConfig(configurable={
        "agent_service": service, "user_id": USER_ID, "thread_id": f"reasoning-demo-{USER_ID}",
    })
    out = await route_intent_node(state, config)
    print(f"\n消息: {msg!r}")
    print(f"  intent={out['intent']}  dest_node={out['dest_node']}  source={out['agent_plan'].get('source')}")
    print(f"  route={out['route']}  memory={len(out['memory_context'])} chars")
    return out

# 规则命中：删除 → resource_delete 子图
out1 = await run_node("删除刚才生成的产品经理JD")
assert out1["dest_node"] == "resource_delete"

# 真实 LLM 路由：闲聊 → chat 节点
out2 = await run_node("你好，你能帮我做些什么？")
assert out2["dest_node"] == "chat"

# skill 意图：邮件 → skill_draft（草稿确认后 send）
out3 = await run_node("给候选人发一封面试邀约邮件")
assert out3["dest_node"] == "skill_draft"
print("\n✅ 三种分发目标验证通过：resource_delete / chat / skill_draft")


消息: '删除刚才生成的产品经理JD'
  intent=resource_delete  dest_node=resource_delete  source=delete_rule
  route=None  memory=0 chars



消息: '你好，你能帮我做些什么？'
  intent=general  dest_node=chat  source=react_router
  route=None  memory=0 chars

消息: '给候选人发一封面试邀约邮件'
  intent=email_notification  dest_node=skill_draft  source=rule_intent
  route=/system/email-configs  memory=0 chars

✅ 三种分发目标验证通过：resource_delete / chat / skill_draft


## §9 清理

关闭真实 DB 会话。本 Notebook 全程只读，无需回滚任何数据。

In [15]:
service._graph_artifacts = []
await db.close()
print("DB 会话已关闭。全程未写库、未删除资源；LLM 调用次数 ≈ 7（§4×4 + §4.1×2 + §8×1）。")

DB 会话已关闭。全程未写库、未删除资源；LLM 调用次数 ≈ 7（§4×4 + §4.1×2 + §8×1）。
